# 02 · Data cleaning & modelling: brand names, missing months, analysis tables  *(v2)*

**Project:** BYD France – Business Intelligence & Market Analytics · **Phase 2 · Notebook 2 of 4**

**v2 changes (after independent review):**
1. `value_type` split into **actual / aggregated / imputed / not_listed**, with separate labels for the monthly and the YTD figure.
2. `source_file` added next to `source_id` on every row, including `not_listed` rows (the file in which the brand was absent).
3. `dim_brand` now separates **brand origin** from **controlling owner country**, with an explicit, documented rule for `is_chinese_owned` (Volvo, MG, joint ventures).
4. **Published CCFA YTD values are preserved.** Revisions are documented in a `ytd_revision_gap` column and never "corrected" with a cumulative sum.

`brand_month_raw.csv` (notebook 01 output) is read only and **not modified**.

### Outputs (`data/processed/`)
| File | Role |
|---|---|
| `brand_month.csv` | Fact table: one row per brand × month (51 brands × 34 months) |
| `market_month.csv` | Fact table: total French new passenger-car market per month |
| `dim_brand.csv` | Dimension table: brand origin, controlling owner, classification flags |

## Data dictionary: value classifications

| `value_type` | Meaning | Example |
|---|---|---|
| **actual** | Copied unchanged from a CCFA PDF | BYD Oct 2026 = 5 419 |
| **aggregated** | Sum of two or more *actual* published lines. No assumption involved | Fiat + Abarth before Oct 2026 (month and published YTD summed) |
| **imputed** | A missing source value filled by a stated rule | Jaguar Jan 2025 YTD: blank in PDF; January YTD = January month |
| **not_listed** | Brand absent from that month's table. Value **unknown, not zero** | Land Rover Mar–Oct 2025 (inside AUTRES) |

* `value_type_month` and `value_type_ytd` label each figure separately. `value_type` is the **row-level label = the least reliable of the two** (order: actual < aggregated < imputed < not_listed). Jaguar Jan 2025 is therefore `month = actual`, `ytd = imputed`, row = `imputed`.
* `share_month_pct` / `share_ytd_pct` are **calculated ratios** (brand ÷ market total, both from the same CCFA table, so the scope matches). Their reliability follows the row's `value_type`.
* `ytd_revision_gap` = published YTD − (previous month's published YTD + this month). **0** = consistent. **≠ 0** = CCFA revised earlier months. **Empty** = cannot be computed (previous month not listed, first month of the series, or YTD imputed). Published YTD is never overwritten.

## Step 1 · Load the extracted data (read only)

**What you should see:** `Rows loaded: 1475 | months: 34`. The fingerprint (MD5) is printed so you can prove at the end that the raw file was not modified.

In [ ]:
import pandas as pd, hashlib
from google.colab import files

uploaded = files.upload()                       # choose brand_month_raw.csv
RAW_NAME = next(iter(uploaded))
RAW_MD5 = hashlib.md5(open(RAW_NAME, "rb").read()).hexdigest()
raw = pd.read_csv(RAW_NAME)
print("Rows loaded:", len(raw), "| months:", raw.period.nunique(), "| raw file MD5:", RAW_MD5)

## Step 2 · Brand dimension: origin vs controlling owner

**Classification methodology (explicit rule):**
* `brand_origin_country`: where the brand was founded / historically based (heritage).
* `controlling_owner`: entity holding **majority (>50%) or sole control** of the brand today. `controlling_owner_country` = its headquarters.
* **`is_chinese_owned` = True only if `controlling_owner_country` = China.**
* **Joint ventures with no single controlling owner** (e.g. 50/50) → `False`, flagged with `is_joint_venture = True`.
* `is_chinese_origin` = brand founded in China (a narrower group).

**Edge cases, decided by the rule:**
| Brand | Origin | Controlling owner | Chinese-owned? | Why |
|---|---|---|---|---|
| **Volvo** | Sweden | Geely Holding (China), majority | ✅ Yes | Majority control is Chinese, although the brand is Swedish |
| **MG** | United Kingdom | SAIC Motor (China), wholly owned | ✅ Yes | British heritage, Chinese owner since 2007 |
| **Smart** | Germany | Mercedes-Benz / Geely, 50/50 JV | ❌ No (JV flag) | No single controlling owner |
| **Leapmotor** | China | Zhejiang Leapmotor (China) | ✅ Yes | Stellantis holds only a minority stake in the brand owner |
| **Mini / Dacia** | UK / Romania | BMW (DE) / Renault (FR) | ❌ No | Shows the same rule applied to non-Chinese owners |

**Data label:** ownership is general industry knowledge as of November 2026, **not CCFA data**. Every row carries a `classification_note`. Re-check it if ownership changes.

**What you should see:** `Brands: 51 | Chinese-owned: 12 | Chinese-origin: 10 | Joint ventures: 1`.

In [ ]:
DIM = [
 # brand, origin, controlling_owner, owner_country, ownership_structure, note
 ("Abarth","Italy","Stellantis","Netherlands","wholly owned","Stellantis is legally domiciled in the Netherlands"),
 ("Alfa Romeo","Italy","Stellantis","Netherlands","wholly owned","Stellantis is legally domiciled in the Netherlands"),
 ("Citroën","France","Stellantis","Netherlands","wholly owned","Stellantis is legally domiciled in the Netherlands"),
 ("DS","France","Stellantis","Netherlands","wholly owned","Stellantis is legally domiciled in the Netherlands"),
 ("Fiat","Italy","Stellantis","Netherlands","wholly owned","Stellantis is legally domiciled in the Netherlands"),
 ("Fiat + Abarth","Italy","Stellantis","Netherlands","wholly owned","Aggregated series of Fiat and Abarth (CCFA merged them in Oct 2026)"),
 ("Jeep","USA","Stellantis","Netherlands","wholly owned","Stellantis is legally domiciled in the Netherlands"),
 ("Lancia","Italy","Stellantis","Netherlands","wholly owned","Stellantis is legally domiciled in the Netherlands"),
 ("Maserati","Italy","Stellantis","Netherlands","wholly owned","Stellantis is legally domiciled in the Netherlands"),
 ("Opel","Germany","Stellantis","Netherlands","wholly owned","Stellantis is legally domiciled in the Netherlands"),
 ("Peugeot","France","Stellantis","Netherlands","wholly owned","Stellantis is legally domiciled in the Netherlands"),
 ("Renault","France","Renault Group","France","wholly owned",""),
 ("Dacia","Romania","Renault Group","France","wholly owned","Romanian-origin brand controlled by France's Renault Group"),
 ("Alpine","France","Renault Group","France","wholly owned",""),
 ("Audi","Germany","Volkswagen Group","Germany","wholly owned",""),
 ("Cupra","Spain","Volkswagen Group","Germany","wholly owned",""),
 ("Porsche","Germany","Volkswagen Group","Germany","majority owned","Porsche AG is listed; Volkswagen AG holds the majority"),
 ("Seat","Spain","Volkswagen Group","Germany","wholly owned",""),
 ("Skoda","Czechia","Volkswagen Group","Germany","wholly owned",""),
 ("Volkswagen","Germany","Volkswagen Group","Germany","wholly owned",""),
 ("BMW","Germany","BMW Group","Germany","wholly owned",""),
 ("Mini","United Kingdom","BMW Group","Germany","wholly owned","British-origin brand controlled by Germany's BMW Group"),
 ("Mercedes-Benz","Germany","Mercedes-Benz Group","Germany","wholly owned",""),
 ("Smart","Germany","Mercedes-Benz Group / Geely Holding (50/50 JV)","Germany / China","joint venture (50/50)",
  "No single controlling owner -> is_chinese_owned = False; flagged as JV"),
 ("Toyota","Japan","Toyota Motor Corporation","Japan","wholly owned",""),
 ("Lexus","Japan","Toyota Motor Corporation","Japan","wholly owned",""),
 ("Hyundai","South Korea","Hyundai Motor Group","South Korea","wholly owned",""),
 ("Kia","South Korea","Hyundai Motor Group","South Korea","group controlled","Kia Corp. is listed; Hyundai Motor Company is the largest shareholder and controls it"),
 ("Ford","USA","Ford Motor Company","USA","wholly owned",""),
 ("Nissan","Japan","Nissan Motor Co.","Japan","independent","Renault-Nissan cross-shareholding below control level"),
 ("Honda","Japan","Honda Motor Co.","Japan","wholly owned",""),
 ("Mazda","Japan","Mazda Motor Corporation","Japan","independent",""),
 ("Suzuki","Japan","Suzuki Motor Corporation","Japan","independent",""),
 ("Mitsubishi","Japan","Mitsubishi Motors","Japan","independent","Nissan holds a significant minority stake"),
 ("Tesla","USA","Tesla, Inc.","USA","independent",""),
 ("Ferrari","Italy","Ferrari N.V.","Italy","independent","Listed company"),
 ("Jaguar","United Kingdom","Tata Motors (JLR)","India","wholly owned","British-origin brand controlled by India's Tata Motors"),
 ("Land Rover","United Kingdom","Tata Motors (JLR)","India","wholly owned","British-origin brand controlled by India's Tata Motors"),
 ("Volvo","Sweden","Geely Holding","China","majority owned",
  "Volvo Cars is listed in Stockholm; Geely Holding holds the majority -> Swedish origin, Chinese controlling owner"),
 ("Lynk & Co","China","Geely Holding","China","group controlled","Chinese brand within the Geely group"),
 ("Zeekr","China","Geely Holding","China","group controlled","Chinese brand within the Geely group"),
 ("Geely","China","Geely Holding","China","wholly owned",""),
 ("BYD","China","BYD Company","China","independent","Subject brand of this project"),
 ("MG","United Kingdom","SAIC Motor","China","wholly owned",
  "British-origin brand, owned by China's state-owned SAIC Motor since 2007 -> Chinese controlling owner"),
 ("Leapmotor","China","Zhejiang Leapmotor Technology","China","independent (minority partner)",
  "Stellantis holds a minority stake (about 21%) in Leapmotor; European sales run through Leapmotor International, a JV majority-held by Stellantis. Brand owner is Chinese -> True"),
 ("Xpeng","China","XPeng Inc.","China","independent",""),
 ("Chery","China","Chery Automobile","China","wholly owned",""),
 ("Omoda","China","Chery Automobile","China","wholly owned","Chery sub-brand"),
 ("Jaecoo","China","Chery Automobile","China","wholly owned","Chery sub-brand"),
 ("Skyworth","China","Skyworth Group","China","group controlled",""),
 ("Other brands (AUTRES)","n/a","n/a","n/a","n/a","Residual of brands not listed separately; composition changes over time"),
]


def build_dim():
    d = pd.DataFrame(DIM, columns=["brand", "brand_origin_country", "controlling_owner", "controlling_owner_country",
                                   "ownership_structure", "classification_note"])
    d["is_chinese_owned"] = d["controlling_owner_country"].eq("China")
    d["is_chinese_origin"] = d["brand_origin_country"].eq("China")
    d["is_joint_venture"] = d["ownership_structure"].str.startswith("joint venture")
    d["is_byd"] = d["brand"].eq("BYD")
    d["classification_rule"] = ("is_chinese_owned = controlling owner (>50% or sole control) headquartered in China; "
                                "50/50 JVs = False; ownership as of Nov 2026, general industry knowledge, not from CCFA")
    return d[["brand", "brand_origin_country", "controlling_owner", "controlling_owner_country", "ownership_structure",
              "is_chinese_owned", "is_chinese_origin", "is_joint_venture", "is_byd", "classification_note",
              "classification_rule"]]

dim_brand = build_dim()
print("Brands:", len(dim_brand), "| Chinese-owned:", dim_brand.is_chinese_owned.sum(),
      "| Chinese-origin:", dim_brand.is_chinese_origin.sum(), "| Joint ventures:", dim_brand.is_joint_venture.sum())
dim_brand[dim_brand.brand.isin(["Volvo", "MG", "Smart", "Leapmotor", "BYD", "Mini", "Dacia"])][
    ["brand", "brand_origin_country", "controlling_owner", "controlling_owner_country", "ownership_structure",
     "is_chinese_owned", "is_chinese_origin", "is_joint_venture"]]

## Step 3 · Map CCFA labels to standard brand names

Same mapping as v1 (e.g. `ALPHA ROMEO` → Alfa Romeo, `M.G.` → MG, `ABARTH/FIAT` → Fiat + Abarth). An unknown label **stops the notebook** rather than being dropped.
**What you should see:** `Labels in data: 52 | unmapped: []`.

In [ ]:
LABEL_MAP = {
    "ABARTH": "Abarth", "ALFA ROMEO": "Alfa Romeo", "ALPHA ROMEO": "Alfa Romeo", "CITROEN": "Citroën", "DS": "DS",
    "FIAT": "Fiat", "JEEP": "Jeep", "LANCIA": "Lancia", "MASERATI": "Maserati", "OPEL": "Opel", "PEUGEOT": "Peugeot",
    "RENAULT": "Renault", "DACIA": "Dacia", "ALPINE": "Alpine", "AUDI": "Audi", "CUPRA": "Cupra", "PORSCHE": "Porsche",
    "SEAT": "Seat", "SKODA": "Skoda", "VOLKSWAGEN": "Volkswagen", "B.M.W.": "BMW", "MINI": "Mini",
    "MERCEDES": "Mercedes-Benz", "SMART": "Smart", "TOYOTA": "Toyota", "LEXUS": "Lexus", "HYUNDAI": "Hyundai",
    "KIA": "Kia", "FORD": "Ford", "NISSAN": "Nissan", "HONDA": "Honda", "MAZDA": "Mazda", "SUZUKI": "Suzuki",
    "MITSUBISHI": "Mitsubishi", "TESLA": "Tesla", "FERRARI": "Ferrari", "JAGUAR": "Jaguar", "LAND ROVER": "Land Rover",
    "VOLVO": "Volvo", "LYNK CO": "Lynk & Co", "ZEEKR": "Zeekr", "GEELY": "Geely", "BYD": "BYD", "M.G.": "MG",
    "LEAPMOTOR": "Leapmotor", "XPENG": "Xpeng", "CHERY": "Chery", "OMODA": "Omoda", "JAECOO": "Jaecoo",
    "SKYWORTH": "Skyworth", "AUTRES": "Other brands (AUTRES)", "ABARTH/FIAT": "Fiat + Abarth",
}

unknown = sorted(set(raw.loc[~raw.is_total_row, "brand_label"]) - set(LABEL_MAP))
print("Labels in data:", raw.loc[~raw.is_total_row, "brand_label"].nunique(), "| unmapped:", unknown)

## Step 4 · Clean and reshape (preserving published YTD)

**What the code does:**
1. Keeps every CCFA figure as published (`actual`).
2. **Jaguar Jan 2025:** the month (0) stays `actual`. The blank YTD is filled with the January rule → `value_type_ytd = imputed`.
3. **Fiat + Abarth:** builds an `aggregated` series by summing the two published lines, **including their published YTDs** (not a running total). October 2026 is CCFA's own combined line → `actual`. The code stops if either component is missing in a month, so an aggregate is never built from half the data.
4. Builds the complete brand × month panel. Absent brands get `not_listed`, empty values, and the `source_id` / `source_file` of the table they were absent from.
5. **`ytd_revision_gap`** documents CCFA revisions. The published YTD itself is never changed.

In [ ]:
def add_ytd_revision_gap(df, key):
    """Published YTD minus (previous month's published YTD + this month). January: YTD minus month.
    Published YTD values are NEVER overwritten; the gap only documents CCFA revisions."""
    df = df.sort_values([key, "period"]).copy()
    df["prev_period"] = (pd.PeriodIndex(df["period"], freq="M") - 1).strftime("%Y-%m")
    prev = df[[key, "period", "registrations_ytd"]].rename(columns={"period": "prev_period", "registrations_ytd": "_prev_ytd"})
    df = df.merge(prev, on=[key, "prev_period"], how="left")
    jan = df["period"].str.endswith("-01")
    expected = df["_prev_ytd"] + df["registrations_month"]
    expected[jan] = df.loc[jan, "registrations_month"]
    df["ytd_revision_gap"] = df["registrations_ytd"] - expected
    return df.drop(columns=["prev_period", "_prev_ytd"])


def clean(raw):
    raw = raw.copy()
    unknown = sorted(set(raw.loc[~raw.is_total_row, "brand_label"]) - set(LABEL_MAP))
    if unknown:
        raise ValueError(f"Unmapped brand labels: {unknown}")

    # ---- market table (published totals, untouched)
    market = (raw[raw.is_total_row][["period", "registrations_month", "registrations_ytd", "source_id", "source_file"]]
              .rename(columns={"registrations_month": "market_registrations_month",
                               "registrations_ytd": "market_registrations_ytd"}).copy())

    # ---- brand rows: start as actual (copied from CCFA)
    b = raw[~raw.is_total_row].copy()
    b["brand"] = b["brand_label"].map(LABEL_MAP)
    b["value_type_month"] = "actual"
    b["value_type_ytd"] = "actual"
    b["note"] = ""

    # Jaguar Jan-2025: month = 0 is actual; YTD blank in PDF -> imputed (January YTD = month by definition)
    jan_blank = b["period"].str.endswith("-01") & b["registrations_ytd"].isna()
    b.loc[jan_blank, "registrations_ytd"] = b.loc[jan_blank, "registrations_month"]
    b.loc[jan_blank, "value_type_ytd"] = "imputed"
    b.loc[jan_blank, "note"] = "YTD cell blank in source PDF; imputed as January YTD = January month"

    # Fiat + Abarth: aggregated series (sum of the two separately published lines, month AND published YTD)
    comp = b[b.brand.isin(["Fiat", "Abarth"])]
    fa = comp.groupby("period", as_index=False).agg(
        registrations_month=("registrations_month", "sum"), registrations_ytd=("registrations_ytd", "sum"),
        source_id=("source_id", "first"), source_file=("source_file", "first"), n_components=("brand", "nunique"))
    if (fa.n_components != 2).any():
        raise ValueError("Fiat + Abarth aggregation expects both components every month: "
                         f"{fa.loc[fa.n_components != 2, 'period'].tolist()}")
    fa = fa.drop(columns="n_components")
    fa["brand"] = "Fiat + Abarth"; fa["brand_label"] = "ABARTH + FIAT"
    fa["value_type_month"] = "aggregated"; fa["value_type_ytd"] = "aggregated"
    fa["note"] = "aggregated: sum of separately published ABARTH and FIAT lines (month and published YTD)"
    merged = b[b.brand_label == "ABARTH/FIAT"].copy()
    merged["note"] = "published by CCFA as one combined ABARTH/FIAT line"
    b = pd.concat([b[b.brand_label != "ABARTH/FIAT"], fa, merged], ignore_index=True)

    # ---- complete panel: every brand x every month; absent = not_listed (missing, never 0)
    periods = sorted(raw.period.unique()); brands = sorted(b.brand.unique())
    grid = pd.MultiIndex.from_product([periods, brands], names=["period", "brand"]).to_frame(index=False)
    panel = grid.merge(b, on=["period", "brand"], how="left")
    panel["listed_in_source"] = panel["registrations_month"].notna()
    nl = ~panel.listed_in_source
    panel.loc[nl, ["value_type_month", "value_type_ytd"]] = "not_listed"
    panel.loc[nl, "note"] = "not listed separately in this month's CCFA table (inside AUTRES or not sold); value unknown, not zero"
    # traceability for absent rows: the file in which the brand was absent
    src = market.set_index("period")[["source_id", "source_file"]]
    panel.loc[nl, "source_id"] = panel.loc[nl, "period"].map(src.source_id)
    panel.loc[nl, "source_file"] = panel.loc[nl, "period"].map(src.source_file)

    # overall row label = least reliable field (not_listed > imputed > aggregated > actual)
    rank = {"actual": 0, "aggregated": 1, "imputed": 2, "not_listed": 3}
    inv = {v: k for k, v in rank.items()}
    panel["value_type"] = [inv[max(rank[m], rank[y])] for m, y in zip(panel.value_type_month, panel.value_type_ytd)]

    # ---- published YTD revisions documented (values unchanged)
    panel = add_ytd_revision_gap(panel, "brand")
    # an imputed YTD cannot be tested against itself -> gap not computable
    panel.loc[panel.value_type_ytd == "imputed", "ytd_revision_gap"] = float("nan")

    # ---- shares (calculated ratios; numerator & denominator from the same CCFA table)
    panel = panel.merge(market[["period", "market_registrations_month", "market_registrations_ytd"]], on="period", how="left")
    panel["share_month_pct"] = (panel.registrations_month / panel.market_registrations_month * 100).round(3)
    panel["share_ytd_pct"] = (panel.registrations_ytd / panel.market_registrations_ytd * 100).round(3)

    panel["in_market_sum"] = panel.listed_in_source & ~panel.brand.isin(["Fiat", "Abarth"])
    panel["year"] = panel.period.str[:4].astype(int)
    panel["month"] = panel.period.str[5:].astype(int)
    panel["date"] = pd.to_datetime(panel.period + "-01")
    cols = ["period", "date", "year", "month", "brand", "brand_label",
            "registrations_month", "registrations_ytd", "ytd_revision_gap",
            "market_registrations_month", "market_registrations_ytd", "share_month_pct", "share_ytd_pct",
            "value_type", "value_type_month", "value_type_ytd", "listed_in_source", "in_market_sum",
            "source_id", "source_file", "note"]
    panel = panel[cols].sort_values(["brand", "period"]).reset_index(drop=True)

    market["brand"] = "TOTAL MARKET"
    market = market.rename(columns={"market_registrations_month": "registrations_month",
                                    "market_registrations_ytd": "registrations_ytd"})
    market = add_ytd_revision_gap(market, "brand").rename(columns={
        "registrations_month": "market_registrations_month", "registrations_ytd": "market_registrations_ytd"})
    market["year"] = market.period.str[:4].astype(int)
    market["month"] = market.period.str[5:].astype(int)
    market["date"] = pd.to_datetime(market.period + "-01")
    market["value_type"] = "actual"
    market = market[["period", "date", "year", "month", "market_registrations_month", "market_registrations_ytd",
                     "ytd_revision_gap", "value_type", "source_id", "source_file"]].sort_values("period").reset_index(drop=True)
    return panel, market

brand_month, market_month = clean(raw)
print("brand_month:", brand_month.shape, "| market_month:", market_month.shape)
print(brand_month.value_type.value_counts().to_string())
pd.crosstab(brand_month.value_type_month, brand_month.value_type_ytd)

## Step 5 · Rerun all validations

| Check | What it proves |
|---|---|
| V1 | Brands (excluding the Fiat/Abarth components) still add up exactly to CCFA's market total, month and YTD |
| V2 | Every `actual` monthly value equals `brand_month_raw.csv` |
| V3 | **Every published YTD is preserved** (brands and market), and each actual row points to the same source file as the raw data |
| V4 | Every `aggregated` value equals the sum of its components |
| V5 | Only the four allowed classifications exist; `not_listed` rows carry no numbers; exactly 1 imputed row |
| V6 | Every row has `source_id` **and** `source_file` |
| V7 | No duplicate brand-month |
| V8 | Every brand has a dimension record, and `is_chinese_owned` follows the documented rule |

**What you should see:** every line `True`, domain = `['actual', 'aggregated', 'imputed', 'not_listed']`, imputed rows = 1.

In [ ]:
def validate(panel, market, raw, dim):
    out = {}
    s = panel[panel.in_market_sum].groupby("period")[["registrations_month", "registrations_ytd"]].sum()
    m = market.set_index("period")
    out["V1 brands sum to market (month)"] = bool((s.registrations_month == m.market_registrations_month).all())
    out["V1 brands sum to market (YTD)"] = bool((s.registrations_ytd == m.market_registrations_ytd).all())

    rb = raw[~raw.is_total_row].copy()
    act = panel[panel.value_type_month == "actual"].merge(
        rb[["period", "brand_label", "registrations_month", "registrations_ytd", "source_id", "source_file"]],
        on=["period", "brand_label"], suffixes=("", "_raw"), how="left")
    out["V2 every actual month equals raw"] = bool((act.registrations_month == act.registrations_month_raw).all())
    a2 = act[act.value_type_ytd == "actual"]
    out["V3 published YTD preserved (no recalculation)"] = bool((a2.registrations_ytd == a2.registrations_ytd_raw).all())
    out["V3 actual rows trace to same source file"] = bool((act.source_file == act.source_file_raw).all())
    rt = raw[raw.is_total_row].set_index("period")
    out["V3 market YTD preserved"] = bool((m.market_registrations_ytd == rt.registrations_ytd).all())

    comp = panel[panel.brand.isin(["Fiat", "Abarth"]) & panel.listed_in_source].groupby("period")[["registrations_month", "registrations_ytd"]].sum()
    agg = panel[(panel.brand == "Fiat + Abarth") & (panel.value_type == "aggregated")].set_index("period")
    out["V4 aggregated = sum of components"] = bool((agg[["registrations_month", "registrations_ytd"]] == comp.loc[agg.index]).all().all())

    out["V5 value_type domain"] = sorted(panel.value_type.unique())
    out["V5 not_listed rows have no values"] = bool(panel.loc[panel.value_type == "not_listed", ["registrations_month", "registrations_ytd"]].isna().all().all())
    out["V5 imputed rows"] = int((panel.value_type == "imputed").sum())
    out["V6 every row has source_id & source_file"] = bool(panel[["source_id", "source_file"]].notna().all().all())
    out["V7 no duplicate brand-month"] = int(panel.duplicated(["period", "brand"]).sum()) == 0
    out["V8 every brand in dim_brand"] = sorted(set(panel.brand) - set(dim.brand)) == []
    out["V8 is_chinese_owned follows rule"] = bool((dim.is_chinese_owned == dim.controlling_owner_country.eq("China")).all())
    return out

results = validate(brand_month, market_month, raw, dim_brand)
for k, v in results.items():
    print(f"{k:<50} {v}")

## Step 6 · Published-YTD revisions (documented, not corrected)

**What it does:** lists every month where CCFA's published YTD differs from *previous published YTD + this month*.
**Cross-check:** the count must equal notebook 01's `ytd_revisions_log.csv` (**52 rows = 37 brand + 15 market**). Same data, same conclusion, computed independently.
**How to read it:** named brands ±1–3 cars and market ±1–5 = routine corrections. AUTRES large negative gaps = brands moved out of "others" and restated.

In [ ]:
brand_rev = brand_month[brand_month.ytd_revision_gap.fillna(0) != 0]
market_rev = market_month[market_month.ytd_revision_gap.fillna(0) != 0]
print("Brand-month revisions:", len(brand_rev), "| market revisions:", len(market_rev), "| total:", len(brand_rev) + len(market_rev))
brand_rev[["period", "brand", "registrations_month", "registrations_ytd", "ytd_revision_gap", "value_type", "source_id"]] \
    .sort_values("ytd_revision_gap", key=abs, ascending=False).head(12)

## Step 7 · Spot-checks on the special cases

* **Jaguar Jan 2025:** `value_type_month = actual`, `value_type_ytd = imputed`, row = `imputed`.
* **Fiat + Abarth:** Aug–Sep 2026 `aggregated`, Oct 2026 `actual`. Sep YTD 24 328 + Oct 2 771 = 27 099 → `ytd_revision_gap = 0`.
* **Land Rover May 2025:** `not_listed`, traced to `Immatriculations-VPN_Mai2025.pdf`.
* **BYD Oct 2026:** 5 419, share 3.629%, `actual`, source SRC037.

In [ ]:
show = ["period", "brand", "registrations_month", "registrations_ytd", "ytd_revision_gap", "share_month_pct",
        "value_type", "value_type_month", "value_type_ytd", "source_id", "source_file"]
display(brand_month.query("brand == 'Jaguar' and period == '2025-01'")[show])
display(brand_month.query("brand == 'Fiat + Abarth' and period >= '2026-08'")[show])
display(brand_month.query("brand == 'Land Rover' and period == '2025-05'")[show])
display(brand_month.query("brand == 'BYD' and period == '2026-10'")[show])

## Step 8 · Export and confirm the raw file is unchanged

**What you should see:** the three shapes, and `Raw file unchanged: True` (same MD5 as Step 1).

In [ ]:
brand_month.to_csv("brand_month.csv", index=False, date_format="%Y-%m-%d")
market_month.to_csv("market_month.csv", index=False, date_format="%Y-%m-%d")
dim_brand.to_csv("dim_brand.csv", index=False)
print("brand_month:", brand_month.shape, "| market_month:", market_month.shape, "| dim_brand:", dim_brand.shape)
print("Raw file unchanged:", hashlib.md5(open(RAW_NAME, "rb").read()).hexdigest() == RAW_MD5)
for f in ["brand_month.csv", "market_month.csv", "dim_brand.csv"]:
    files.download(f)

## Summary

* **Classifications:** 1 440 actual · 33 aggregated · 1 imputed · 260 not_listed (1 734 rows).
* **Traceability:** every row carries `source_id` + `source_file`.
* **Ownership:** rule-based `is_chinese_owned` (12 brands), separate from `is_chinese_origin` (10), with the joint venture flagged.
* **YTD integrity:** published values preserved. 52 revisions documented, consistent with notebook 01.

**Next:** notebook 03, exploratory analysis.